In [ ]:

import pandas as pd
from pathlib import Path
import json


ROOT = Path.cwd().parents[1]     
DATA_SOURCE = 'env_wasmun'
DATA_RAW_DIR = ROOT / "data/raw" / DATA_SOURCE     # notebooks/python -> project root
EXPORT_DIR = ROOT / "data/interim" 
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_FILE = EXPORT_DIR / f"{DATA_SOURCE}.parquet"
df = pd.read_parquet(DATA_RAW_DIR / "df.parquet")
codebook = json.load(open(DATA_RAW_DIR / "lookup.json"))

df.head()


    

,freq,wst_oper,unit,geo,time,value
0,A,GEN,KG_HAB,BE,2010,456.0
1,A,GEN,KG_HAB,BE,2011,455.0
2,A,GEN,KG_HAB,BE,2012,445.0
3,A,GEN,KG_HAB,BE,2013,436.0
4,A,GEN,KG_HAB,BE,2014,425.0


In [24]:
#Inspect the metadata of env_wasmun that is saved in the parquet file
codebook

{'freq': {'A': 'Annual'},
 'wst_oper': {'GEN': 'Waste generated',
  'TRT': 'Waste treatment',
  'DSP_I_RCV_E': 'Disposal - incineration (D10) and recovery - energy recovery (R1)',
  'DSP_L_OTH': 'Disposal - landfill and other (D1-D7, D12)',
  'DSP_I': 'Disposal - incineration (D10)',
  'RCV_E': 'Recovery - energy recovery (R1)',
  'RCY': 'Recycling',
  'RCY_M': 'Recycling - material',
  'RCY_C_D': 'Recycling - composting and digestion',
  'PRP_REU': 'Preparing for reuse'},
 'unit': {'KG_HAB': 'Kilograms per capita', 'THS_T': 'Thousand tonnes'},
 'geo': {'BE': 'Belgium',
  'BG': 'Bulgaria',
  'CZ': 'Czechia',
  'DK': 'Denmark',
  'DE': 'Germany',
  'EE': 'Estonia',
  'IE': 'Ireland',
  'EL': 'Greece',
  'ES': 'Spain',
  'FR': 'France',
  'HR': 'Croatia',
  'IT': 'Italy',
  'CY': 'Cyprus',
  'LV': 'Latvia',
  'LT': 'Lithuania',
  'LU': 'Luxembourg',
  'HU': 'Hungary',
  'MT': 'Malta',
  'NL': 'Netherlands',
  'AT': 'Austria',
  'PL': 'Poland',
  'PT': 'Portugal',
  'RO': 'Romania',
  'SI

In [44]:
#check the value geo is DE and wst_oper is GEN

df[(df['geo'] == 'DE') & (df['unit'] == 'KG_HAB') & (df['wst_oper'] == 'RCY')]

,freq,wst_oper,unit,geo,time,value
6900,A,RCY,KG_HAB,DE,2010,376.0
6901,A,RCY,KG_HAB,DE,2011,394.0
6902,A,RCY,KG_HAB,DE,2012,404.0
6903,A,RCY,KG_HAB,DE,2013,392.0
6904,A,RCY,KG_HAB,DE,2014,414.0
6905,A,RCY,KG_HAB,DE,2015,422.0
6906,A,RCY,KG_HAB,DE,2016,425.0
6907,A,RCY,KG_HAB,DE,2017,421.0
6908,A,RCY,KG_HAB,DE,2018,407.0
6909,A,RCY,KG_HAB,DE,2019,406.0


In [ ]:
# Each indicator is one combination of `unit` and `wst_oper`.
# select_indicator() keeps that combination and names the value column after the indicator.

def select_indicator(df, unit, wst_oper, name):
    """Return geo, time and the value for one unit and waste operation, with value renamed to `name`."""
    rows = (df['unit'] == unit) & (df['wst_oper'] == wst_oper)
    return df.loc[rows, ['geo', 'time', 'value']].rename(columns={'value': name})


df_gen_pc    = select_indicator(df, 'KG_HAB', 'GEN', 'municipal_waste_gen_pc')     # kg per person
df_gen_total = select_indicator(df, 'THS_T',  'GEN', 'municipal_waste_gen_total')  # thousand tonnes
df_rcy_total = select_indicator(df, 'THS_T',  'RCY', 'municipal_waste_rcy_total')  # thousand tonnes

# One row per geo-year. Left join on recycling, so years without recycling data are kept.
df_merged = (
    df_gen_pc
    .merge(df_gen_total, on=['geo', 'time'])
    .merge(df_rcy_total, on=['geo', 'time'], how='left')
)

# Recycling rate = recycled / generated (RCY = material recycling + composting and digestion)
df_merged['municipal_waste_rcy_rate'] = (
    df_merged['municipal_waste_rcy_total'] / df_merged['municipal_waste_gen_total']
)


In [ ]:
#EXPORT FILE
df_merged.to_parquet(EXPORT_FILE, index=False)